In [2]:
import torch
import torch.nn as nn

In [2]:
# tensor là số chiều = số lớp [] lồng nhau
a = torch.tensor([1,2,3])
b = torch.tensor([2,3,4])
print(a.shape)

torch.Size([3])


In [11]:
# các phép toán vấn đúng với tensor
# các số thông thường sẽ tự động mở rộng để phù hợp với tensro
# nhưng nó vẫn ko phải tensor nó vẫn là 2
print (a+2)
print (a/2)
print (a*2)

# các phép toán tensor với nhau
# phần tử tại các vị trí thực hiện phép tính với nhau
print (a*b)

tensor([3, 4, 5])
tensor([0.5000, 1.0000, 1.5000])
tensor([2, 4, 6])
tensor([ 2,  6, 12])


In [ ]:
# thử tính gradient cho phương trình y theo x và z
x = torch.tensor(3.0, requires_grad = True)
z = torch.tensor(2.0, requires_grad = True)
y = x**2 + 2*z*x + 3

y.backward() # tính luôn theo cả x và z

print(x.grad , z.grad) # cái này chỉ để in ra kết quả đã tính

tensor(10.) tensor(6.)


In [ ]:
# hàm tính toán cho tensor
def tinhtoan(c):
    return c.sum(), c.mean()

a = a.float()
print (tinhtoan(a)) 

(tensor(6.), tensor(2.))


In [ ]:
# tính tích vô hướng của các tensor
print(torch.dot(a,b))

tensor(20)


TENSOR CƠ BẢN

In [ ]:
# ngẫu nhiên các giá trị nguyên từ 0-100 cho 1 tensor có (3,4,5,6) 
d = torch.randint(0,100, (3,4,5,6))

# ngẫu nhiên các giá trị số thực cho tensor có (3,4,5,6)
e = torch.rand((2,3,4,5)) 
print(e) # giá trị trong tensor chỉ từ 0-1

# in dtype của giá trị trong tensor
print(d.dtype, e.dtype)

# in device đang tính toán
print(d.device)

# đọc shape của tensor
print(e.shape) # đọc từ ngoài vào trong

# đổi thứ tự các chiều
print(e.permute(2,1,0,3)) # đổi theo vị trí index của các chiều tensor cũ
# các hiểu khá dễ:
    # tensor "đọc từ ngoài vào" == theo chiều .shape in ra
    # "đọc từ trong ra" == ngược chiều .shape in ra
    # dàn các giá trị ra theo đường trải phẳng "đọc từ trong ra" 
    # gom chúng lại theo chiều "đọc từ trong ra"
        # (4,3,2) gom lần lượt 2 giá trị 1 theo trải phẳng
                # gom lần lượt 3 cụm "2 giá trị bên trên"
                # sau đó gom 4
                
# tính giá trị trung bình từng channel
def trungbinh(n, channel):
    n= n.float()
    return n[:,channel,:,:].mean()

# flatten tensor gốc thành còn có 2 dim
e1 =e.flatten(1,2) # gộp từ dim có index từ 1 đến hết 2 
# giờ là (2,3*4,5) = (2,12,5)
print(e1) # kiểu tháo cái [] ra cho mấy con số nó lại gần nhau theo 

# thêm 1 dim cho 1 tensor
e2 = e.unsqueeze(4) # có thể tạo ở dim mìn muốn
print(e2)

BROADCASTING VÀ PHÉP TOÁN
+ broadcasting nó là dạng cơ chế cho phép 2 tensor khác shape có thể thực hiện phép tính với nhau bằng cách tự mở rộng tensor


In [ ]:
g = torch.rand((2,5,2,2))
print(g)

g1 = torch.rand((5,1,2))
print(g1)

# cái có tensor nhỏ hơn phải thỏa mãn điều kiện:
    # dim tương đương với dim của tensor lớn phải bằng hoặc bằng 1
    # đây là điều kiện bắt buộc nếu ko ko thể chạy
    # giá trị dim là số phần tử trong dim đó:
        # nếu nó là 1 thì có thể tạo thêm n lần để bằng dim tensor lớn
        # nếu nó bằng thì thêm vào đằng trước 1 dim nữa để thêm n lần 
print(g1 + g)

sample = torch.rand((10))
loss = torch.rand((10))
def trungbinhweightloss(a,b):
    return sum(a*b)/sum(a)

x = torch.rand((3,32,32))
mean = (torch.ones((3))*0.5).reshape(3,1,1)
std = (torch.ones((3))*0.5).reshape(3,1,1)
print((x-mean)/std)

y = torch.rand((3,32,32))
def cosin(x, y):
    x= x.flatten(0,x.ndim-1) #ndim là số dim của tensor
    y = y.flatten(0,y.ndim-1)
    dot = torch.dot(x,y)
    x_square = torch.sqrt(torch.dot(x,x))
    y_square = torch.sqrt(torch.dot(y,y))
    return dot/(x_square*y_square)

print(cosin(x,y))

AUTO GRAD

In [ ]:
# thông báo cho pytorch biết là theo dõi grad theo x
x = torch.tensor(2.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)

# trình y theo biến x, z được theo dõi
y = (x-3)**2 

# tính grad của y theo các biến được theo dõi
y.backward() # kết quả được lưu vào .grad

# in ra kết quả đã tính ở bên trên
def giatri(n, lr,x,y,m):
    # n là số vòng lặp
    # m là giá trị đầu tiên của x
    # y là phương trình mà người dùng muốn tìm
    m = float(input("nhập số bắt đầu m"))
    x = torch.tensor(m, requires_grad=True)
    minn = (m-3)**2
    x_minn = x
    for i in range(n):
        y = (x-3)**2
        y.backward()
        with torch.no_grad():
            x -= lr*x.grad
        x.grad.zero_()
        if minn > y:
            minn = y.item()
            x_minn = x.item()
        if i%10 == 0:
            print(x.item())
    return x_minn, minn

# hoặc dùng thủ công hàm optimizer 
def optimize_sgd(n,m):
    x = torch.tensor(2.0, requires_grad=True)
    optimize = torch.optim.SGD([x], lr=0.1)
    minn = (x-3)**2
    minn = minn.item()
    x_minn = x.item()
    for i in range(n):
        optimize.zero_grad()
        y = (x-3)**2
        y.backward()
        optimize.step()
        if minn > y.item():
            minn = y.item()
            x_minn = x.item()
    return minn, x_minn

# detach() chặn lan truyền gradient của đúng 1 tensor
# cái nào gọi .detach() ra thì bị chặn ko cho tính  
x1 = torch.tensor(2.0, requires_grad=True)
y = x1*3
z= x1+ y.detach() # chặn mỗi y trong z /\ z,x1 vẫn tính
k =2*y
loss = z*2 + 2*k # chặn mỗi y trong z còn lại vẫn tính
loss.backward()
print(x1.grad)

# with torch.no_grad() đi chặn những công thức có trong hàm thôi
x2 = torch.tensor(3.0, requires_grad=True)
y= x2**2
z= 2*y
with torch.no_grad(): 
    k = 2*y
    h = 3*z
k1 = 2*k    # torch.no_grad chặn liên kết giữa h và y
h1 = 2*h

k = 2*y     # cài này vẫn được bình thường 
h = 3*z
# k1.backward()
h.backward(retain_graph=True)
k.backward(retain_graph=True)
# h1.backward()
print(x2.grad)

# optimizer.zero_grad()
optimi = torch.optim.SGD([x2], lr=0.1)
for i in range(10):
    optimi.zero_grad() # reset lại cái gradient cũ
    y = x2**4 + 4*x2**3 + 4
    y.backward()    # tính gradient mới 
    optimi.step()   # weight = weightcu - lr.gradient mới
    print(x2.item())


XÂY MODEL

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.fc1 = nn.Linear(728,128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128,10)
    
    def forward(self,x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x) # dropout nên để sau activation
        x = self.fc2(x)
        return x

model = MLP()
print(model)
for i in model.parameters():
    print(i.shape)
# cái này in ra khá là hay
# in ra weight và bias:
""" fc1.weight
    fc1.bias
    fc2.weight
    fc2.bias
"""

x = torch.rand(1, 728)

# train model # nhìn class nó mới chỉ có quá trình forward
# nên train mới chỉ là gán weight random rồi đưa data đi qua
# rồi ra output
# có các layer fc, relu còn cả dropout để random loại % weight
model.train() # tín hiệu train model: forward pass
out1 = model(x) # data đi qua các lớp đưa ra output có dropout
out2 = model(x) #

# tính eval # mới có quá trình forward 
# nên eval mới chỉ là random weight khởi đầu rồi đưa ra output
# dopout sẽ bị ngắt khi tính eval , ko như train
model.eval()  # tín hiệu tính eval: input đi qua các lớp
outtt1 = model(x) # weight chung, ko dopout nên cùng output t1,t2
outtt2 = model(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv2dv1 = nn.Conv2d(
            in_channels= 3,
            out_channels= 16,
            kernel_size= 3
        )
        self.reluv1 = nn.ReLU()
        self.maxpool2dv1 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
        self.conv2dv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3
        )
        self.reluv2 = nn.ReLU()
        self.maxpool2dv2 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
        self.fc1 = nn.LazyLinear(10)
    
    def forward(self,x):
        x = self.conv2dv1(x)
        x = self.reluv1(x)
        x = self.maxpool2dv1(x)
        x = self.conv2dv2(x)
        x = self.reluv2(x)
        x = self.maxpool2dv2(x)
        x = x.flatten(1)
        x = x.fc1(x)

LOSS VÀ OPTIMIZER

In [ ]:
# output này là output từ model bên trên khi đưa data qua
# tạo 1 ouput với kích thước giả
# tạo label cho cái output đó
# tính loss dựa trên công thức crossentropy
output1 = torch.rand((4,2)) # thực thế thì output này là có sẵn
print(output1) # output gồm batch/mẫu và class/lớp
# mẫu, batch, dòng là số lượng ảnh đưa vào model cùng 1 lúc (dùng ảnh cho dễ hiểu)
# class chứa giá trị tỉ lệ model dự đoán nó thuộc class đó (chó, mèo, mỗi loại bao nhiêu %)

label1 = torch.randint(0,2, (4,)) # label cho class và mẫu (đọc đúng thứ tự)
print(label1) # mẫu ở output tương ứng với số label đề là 4
# số lượng class là 2 tương tưng ứng với [0,2) là 0 hoặc 1 trong label
# có thể là chó hoặc mèo  hoặc ô tô, xe máy,....
# nói chung class trong label chỉ nói lên model phải phân loại bao nhiêu lớp
# quan trọng vẫn là cái giá trị của class trong output gán cho nó tính theo %

output2 = torch.rand((4,2))
print(output2)
label2 = torch.randint(0,2,(4,))
print(label2)

lossfn = nn.CrossEntropyLoss() 
# đi qua softmax để chuyển tổng các dự đoán trong output thành 1
# chuẩn bị nhãn thực tế (one hot encoding/ nhiều pp khác)
# ảnh thực tế nhãn là 1, còn lại là 0 hết
# tính cross entropy loss cho tensor batch output (có công thức) 
# giữa nhãn và ln(output softmax)
loss1 = lossfn(output1, label1)
loss2 = lossfn(output2, label2) 

print(loss1, loss2)

In [13]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.fc1 = nn.Linear(64,32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32,16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16,4)
        
        self.lossfnn = nn.CrossEntropyLoss()
        
    def forward(self,x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x
        
    def compute_loss(self,x, label):
        losss = self.lossfnn(x, label)
        return losss

# khởi tạo model và optimizer
model = mlp()
optimii = torch.optim.Adam(model.parameters(), lr = 0.01)

# tạo dữ liệu đầu vào 
x = torch.rand((8,64), requires_grad=True)
label = torch.randint(0,4 ,(8,))

# bật chế độ train trước khi chạy dữ liệu
model.train()

# output lan truyền xuôi
output = model.forward(x)

# tính loss
kq_loss = model.compute_loss(output, label)

# in trọng số trước bước step()
w_befor = model.fc1.weight[0,0].clone().item()
print(w_befor)

# lan truyền ngược và cập nhật trọng số
optimii.zero_grad() # xóa gradient cũ
kq_loss.backward()  # tính gradient từ loss
optimii.step()       # cập nhật trọng số của model

# in trọng số sau bước step()
w_after = model.fc1.weight[0,0].clone().item()
print(w_after)

0.009232789278030396
-0.0007671313360333443
